# Isaac Sim PointNav / ObjectNav Benchmarks for AgenticMemoryNav

This notebook explains how to run and evaluate the two Isaac Sim navigation
benchmarks added to this project:

- **PointNav**: a lightweight, procedurally generated point-goal navigation
  benchmark driven directly by `IsaacSimExecutor` (no external scene/episode
  dataset download required).
- **ObjectNav**: object-goal navigation over InteriorAgent-style USD scenes,
  driven by the full `NavigationPipeline` (planner + scene graph + memory),
  requiring the external InteriorAgent dataset.

See [docs/dependency-decisions.md](../docs/dependency-decisions.md) and
[docs/limitations.md](../docs/limitations.md) for verified facts and fidelity
caveats behind these numbers -- they are **not** paper-comparable Habitat
PointNav/ObjectNav leaderboard results.

This notebook uses the **plain Python 3 kernel** (the project's own venv) and
shells out to Isaac Sim's bundled Python for every simulation step, since Isaac
Sim's `SimulationApp` can only be created once per process and this project's
own dependencies (matplotlib, etc.) are simplest to use from the regular venv.

## 1. Prerequisites and environment checks

- A local Isaac Sim install (verified with 6.0.1) at `~/isaacsim`.
- An NVIDIA GPU with a working driver (`nvidia-smi`).
- Isaac Sim's bundled Python must have this project installed, plus `networkx`
  (its only missing core dependency):

```bash
conda deactivate  # avoid conda's Python shadowing Isaac Sim's own interpreter
~/isaacsim/kit/python/bin/python3 -m pip install -e .
~/isaacsim/kit/python/bin/python3 -m pip install networkx
```

Run the checks below first.

In [1]:
import json
import os
import subprocess
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ISAACSIM_ROOT = Path(os.path.expanduser("~/isaacsim"))
ISAACSIM_PYTHON = ISAACSIM_ROOT / "python.sh"

print("Repo root:", REPO_ROOT)
print("Isaac Sim root exists:", ISAACSIM_ROOT.exists())
print("Isaac Sim python.sh exists:", ISAACSIM_PYTHON.exists())

gpu_check = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"],
    capture_output=True,
    text=True,
)
print(gpu_check.stdout or gpu_check.stderr)


Repo root: /home/snt/projects/AgenticMemoryNav
Isaac Sim root exists: True
Isaac Sim python.sh exists: True
name, driver_version, memory.total [MiB]
NVIDIA RTX A6000, 595.71.05, 49140 MiB



In [2]:
# Confirms Isaac Sim's bundled Python has this project + its dependencies importable.
# First run is slow (Isaac Sim shader/kernel cache is cold); subsequent runs are much faster.
check = subprocess.run(
    [str(ISAACSIM_PYTHON), "-c", "import agentic_memory_nav, networkx; print('env-ready')"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(check.stdout[-2000:])
print(check.stderr[-2000:])

env-ready




## 2. PointNav: generate episodes and run the harness

`src/agentic_memory_nav/datasets/pointnav.py` generates point-goal episodes
procedurally over a fixed obstacle layout (matching `IsaacSimExecutor`'s
default scene), using `src/agentic_memory_nav/mapping/occupancy_grid.py` to
compute geodesic shortest-path distances for SPL. `configs/isaacsim_pointnav.yaml`
controls episode count, seed, and success threshold.

`scripts/run_isaacsim_pointnav.py`, for each episode:
1. Teleports the robot to the episode start.
2. Repeatedly calls `IsaacSimExecutor.send_waypoint` toward the goal until the
   robot is within `success_threshold_m` or `max_steps_per_episode` is reached.
3. Computes `success` (distance-based) and `spl` (`success_weighted_path_length`)
   from the shortest and actual path lengths.

**Limitation**: movement is kinematic (direct pose interpolation) and never
reports a real collision, so SPL will read close to 1.0 even with obstacles
present in the occupancy grid -- this exercises the harness and executor
correctly, but is not a true obstacle-avoidance benchmark yet.

In [3]:
pointnav_run = subprocess.run(
    [
        str(ISAACSIM_PYTHON),
        "scripts/run_isaacsim_pointnav.py",
        "--config",
        "configs/isaacsim_pointnav.yaml",
    ],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(pointnav_run.stdout[-3000:])
if pointnav_run.returncode != 0:
    print(pointnav_run.stderr[-3000:])


ntnav.yaml']
Warp 1.13.0 initialized:
   CUDA Toolkit 12.9, Driver 13.2
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA RTX A6000" (47 GiB, sm_86, mempool enabled)
   Kernel cache:
     /home/snt/.cache/warp/1.13.0

`SimulationApp` class has been instantiated. It is a requirement of the
Carbonite framework's extension/runtime plugin system.

Ensure that the `SimulationApp` class is instantiated before importing
any other Omniverse/Isaac Sim modules, as shown below:

    ------------------------------------------------------------------
    from isaacsim import SimulationApp

    # instantiate the SimulationApp helper class
    simulation_app = SimulationApp({"headless": False})

    # execute other Omniverse/Isaac Sim imports after instantiating it
    from isaacsim...
    ------------------------------------------------------------------



`SimulationApp` class has been instantiated. It is a requirement of the
Carbonite framework's extension/runtime plugin system.

E

In [4]:
# Locate the run directory printed above ("Run artifacts: ...") and inspect results.
run_dir_lines = [
    line for line in pointnav_run.stdout.splitlines() if line.startswith("Run artifacts:")
]
pointnav_run_dir = Path(run_dir_lines[0].split(": ", 1)[1]) if run_dir_lines else None
print("PointNav run dir:", pointnav_run_dir)

if pointnav_run_dir is not None:
    metrics = json.loads((pointnav_run_dir / "metrics.json").read_text())
    episodes = json.loads((pointnav_run_dir / "episodes.json").read_text())
    print(json.dumps(metrics, indent=2))
    summary_rows = [(e["episode_id"], e["success"], round(e["spl"], 3)) for e in episodes]
    print(f"\n{len(episodes)} episodes: ", summary_rows)


PointNav run dir: /home/snt/projects/AgenticMemoryNav/outputs/20260813T210519Z_ddc79c9b
{
  "num_episodes": 10,
  "run_id": "20260813T210519Z_ddc79c9b",
  "spl": 1.0,
  "success_rate": 1.0
}

10 episodes:  [('pointnav_0000', True, 1.0), ('pointnav_0001', True, 1.0), ('pointnav_0002', True, 1.0), ('pointnav_0003', True, 1.0), ('pointnav_0004', True, 1.0), ('pointnav_0005', True, 1.0), ('pointnav_0006', True, 1.0), ('pointnav_0007', True, 1.0), ('pointnav_0008', True, 1.0), ('pointnav_0009', True, 1.0)]


### Interpreting PointNav results

- `metrics.json` -> `success_rate`: fraction of episodes where the robot ended
  within `success_threshold_m` of the goal.
- `metrics.json` -> `spl` (Success weighted by Path Length): mean of
  `success * shortest_path / max(shortest_path, actual_path)` per episode --
  1.0 means every successful episode took the shortest possible route.
- `episodes.json`: per-episode `shortest_path_m` (geodesic, from the occupancy
  grid), `path_length_m` (actual distance travelled), and `final_distance_to_goal_m`.
- `trajectory.jsonl` in the run directory: per-step robot position and
  distance-to-goal for every episode, useful for plotting a single episode's path.

## 3. ObjectNav: download InteriorAgent and run one experiment

ObjectNav uses the InteriorAgent dataset
(https://huggingface.co/datasets/spatialverse/InteriorAgent) and an
`experiments.json` file describing tasks in the schema documented in
[docs/dependency-decisions.md](../docs/dependency-decisions.md) (adapted from
the publicly documented format of
`github.com/learnsyslab/isaac-objnav-semistatic-eval`; no code from that
repository is used here -- it has no published license, see
docs/dependency-decisions.md for the check).

### Download InteriorAgent

```bash
pip install -U "huggingface_hub[cli]"
huggingface-cli login   # only needed if the dataset requires authentication
huggingface-cli download spatialverse/InteriorAgent --repo-type dataset --local-dir ~/data/InteriorAgent
```

Then provide your own `experiments.json` (or one distributed with the dataset)
describing scenes and goals, following the schema in `src/agentic_memory_nav/datasets/objectnav.py`.

**Status**: this dataset was not available in the environment used to build this
notebook, so the cell below is provided for you to run once you have downloaded
it -- it has not been executed end-to-end here. `IsaacSimObjectNavExecutor.shortest_distance_to_goal()`
currently reports a Euclidean lower-bound distance (see docs/limitations.md), not
a validated geodesic distance.

In [5]:
# Fill these in once InteriorAgent is downloaded, then run this cell.
interior_agent_root = os.environ.get("INTERIOR_AGENT_ROOT", "")
experiments_json = os.environ.get("OBJECTNAV_EXPERIMENTS_JSON", "")
experiment_name = os.environ.get("OBJECTNAV_EXPERIMENT_NAME", "")

if interior_agent_root and experiments_json and experiment_name:
    objectnav_run = subprocess.run(
        [
            str(ISAACSIM_PYTHON), "scripts/run_isaacsim_objectnav.py",
            "--scene-root", interior_agent_root,
            "--experiments-json", experiments_json,
            "--experiment", experiment_name,
        ],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    print(objectnav_run.stdout[-3000:])
    if objectnav_run.returncode != 0:
        print(objectnav_run.stderr[-3000:])
else:
    print("Set INTERIOR_AGENT_ROOT, OBJECTNAV_EXPERIMENTS_JSON, and \
OBJECTNAV_EXPERIMENT_NAME env vars.")


Set INTERIOR_AGENT_ROOT, OBJECTNAV_EXPERIMENTS_JSON, and OBJECTNAV_EXPERIMENT_NAME env vars.


### Interpreting ObjectNav results

`objectnav_metrics.json` (in the printed run directory) reports:

- `distance_to_goal_m`: distance from the robot's final position to the
  nearest matching goal asset when the run ended.
- `success`: whether `distance_to_goal_m <= success_threshold_m`.
- `pipeline_success_rate` / `pipeline_spl`: the same metrics the mock end-to-end
  demo reports, computed by the full `NavigationPipeline` (planner, scene graph,
  memory) driving `IsaacSimObjectNavExecutor` for this experiment.

The run directory also contains the usual pipeline artifacts
(`scene_graph.json`, `memory_snapshot.json`, `trajectory.jsonl`, `logs.jsonl`)
so you can inspect what the planner detected and how it decided to explore or
navigate toward the target label.

## 4. Troubleshooting

- **`ModuleNotFoundError: No module named 'agentic_memory_nav'` when running via `python.sh`**:
  re-run `~/isaacsim/kit/python/bin/python3 -m pip install -e .` from the repo root.
- **`Warning: running in conda env, please deactivate before executing this script`**:
  run `conda deactivate` in the shell before invoking `python.sh` (a notebook's
  own kernel environment does not affect the subprocess's inherited shell env,
  but a terminal you launch Jupyter from might still have conda active).
- **First Isaac Sim launch takes 1-2 minutes**: this is normal (shader/kernel
  cache compilation); later launches in the same environment take ~10-25s.
- **No `metrics.json`/final `print()` output after a script finishes**:
  `SimulationApp.close()` terminates the hosting process, so results must be
  written/printed *before* calling `executor.close()` -- both benchmark scripts
  already do this, but keep it in mind if you customize them.
- **`isaacsim is not importable` errors**: confirm you are invoking
  `~/isaacsim/python.sh`, not a plain `python3`/the project's own venv.
- **GPU not visible / `nvidia-smi` fails**: Isaac Sim and its rendering
  pipeline require a working NVIDIA driver; verify with `nvidia-smi` before
  filing an Isaac Sim issue.